In [1]:
class ExtendedEnv:
    def __init__(self):
        self.state = 0
        self.end_state = 5
        self.states = list(range(0, self.end_state + 1))  # tous les états
        self.actions = [0, 1]  # 0 = reculer, 1 = avancer

    def reset(self):
        self.state = 0
        return self.state

    def step(self, action):
        if action == 1 and self.state < self.end_state:
            self.state += 1
        elif action == 0 and self.state > 0:
            self.state -= 1

        reward = 1 if self.state == self.end_state else 0
        done = (self.state == self.end_state)
        return self.state, reward, done, {}

    def get_all_states(self):
        return self.states

    def get_all_actions(self):
        return self.actions


In [2]:
import random

def generate_episode(env, policy):
    episode = []
    state = env.reset()
    done = False

    while not done:
        actions = list(policy[state].keys())
        probs = list(policy[state].values())
        action = random.choices(actions, weights=probs, k=1)[0]  # TODO understand
        next_state, reward, done, _ = env.step(action)
        episode.append((state, action, reward))
        state = next_state

    return episode

In [ ]:
from types import LambdaType
def on_policy_fisrt_visit_mc_control(env, epsilon, num_episodes):
  # Get all states and actions from env
  all_states = env.get_all_states()
  all_actions = env.get_all_actions()
  gamma = 0.99

  policy = {}
  for state in all_states:
    policy[state] = {}
    for action in all_actions:
      policy[state][action] = epsilon / len(all_actions)

  Q = {}
  for state in all_states:
    Q[state] = {}
    for action in all_actions:
      Q[state][action] = 0.0  # valeur initiale arbitraire (zéro ici)

  Returns = {}
  for state in all_states:
    for action in all_actions:
      Returns[(state, action)] = []

  for episode_num in range(num_episodes):
    episode = generate_episode(env, policy)
    G = 0
    i = len(episode) - 1
    while i >= 0:
      state, action, reward = episode[i]
      G = gamma * G + reward

      # Check if (state, action) is first visit in this episode
      first_occurrence = (state, action) not in [(ep[0], ep[1]) for ep in episode[0:i]]

      if first_occurrence:
        Returns[(state, action)].append(G)
        Q[state][action] = sum(Returns[(state, action)]) / len(Returns[(state, action)])

        max_action = max(Q[state], key=Q[state].get)
        for a in Q[state]:
            if a == max_action:
                policy[state][a] = 1 - epsilon + epsilon / len(all_actions)
            else:
                policy[state][a] = epsilon / len(all_actions)
      i -= 1

  return policy,Q


In [6]:
env = ExtendedEnv()
print(on_policy_fisrt_visit_mc_control(env, 0.1, 10000))

({0: {0: 0.05, 1: 0.9500000000000001}, 1: {0: 0.05, 1: 0.9500000000000001}, 2: {0: 0.05, 1: 0.9500000000000001}, 3: {0: 0.05, 1: 0.9500000000000001}, 4: {0: 0.05, 1: 0.9500000000000001}, 5: {0: 0.05, 1: 0.05}}, {0: {0: 0.9453537887852539, 1: 0.9561590552230605}, 1: {0: 0.9457710785805928, 1: 0.96693614555338}, 2: {0: 0.9562184085860866, 1: 0.9778270960590993}, 3: {0: 0.9666147941947419, 1: 0.988887466263095}, 4: {0: 0.9778785147358404, 1: 1.0}, 5: {0: 0.0, 1: 0.0}})
